# 评估器

在高层次上，评估器根据参考示例判断您的大语言模型应用程序的调用，并返回评估分数。

在 LangSmith 评估器中，我们将这个过程表示为一个函数，它接收一个 Run（表示大语言模型应用程序调用）和一个 Example（表示要评估的数据点），并返回 Feedback（表示评估器对大语言模型应用程序调用的评分）。

![Evaluator](../../images/evaluator.png)

这里是一个非常简单的自定义评估器示例，它将模型的输出与数据集中的预期输出进行比较：

In [ ]:
from langsmith.schemas import Example, Run

def correct_label(inputs: dict, reference_outputs: dict, outputs: dict) -> dict:
  score = outputs.get("output") == reference_outputs.get("label")
  return {"score": int(score), "key": "correct_label"}

### 大语言模型评判员评估

大语言模型评判员评估器使用大语言模型来评分系统输出。要使用它们，您通常在大语言模型提示词中编码评分规则/标准。它们可以是无参考的（例如，检查系统输出是否包含攻击性内容或是否符合特定标准）。或者，它们可以将任务输出与参考进行比较（例如，检查输出相对于参考是否事实准确）。

这里是如何使用结构化输出定义大语言模型评判员评估器的示例

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()

class Similarity_Score(BaseModel):
    similarity_score: int = Field(description="1到10之间的语义相似性分数，其中1表示不相关，10表示完全相同。")

# 注意：这是我们的评估器
def compare_semantic_similarity(inputs: dict, reference_outputs: dict, outputs: dict):
    input_question = inputs["question"]
    reference_response = reference_outputs["output"]
    run_response = outputs["output"]
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {   
                "role": "system",
                "content": (
                    "您是一个语义相似性评估器。比较两个问题回答的含义，"
                    "参考回答和新回答，其中参考是正确答案，我们试图判断新回答是否相似。"
                    "提供1到10之间的分数，其中1表示完全不相关，10表示含义完全相同。"
                ),
            },
            {"role": "user", "content": f"问题: {input_question}\n 参考回答: {reference_response}\n 运行回答: {run_response}"}
        ],
        response_format=Similarity_Score,
    )

    similarity_score = completion.choices[0].message.parsed
    return {"score": similarity_score.similarity_score, "key": "similarity"}

让我们试试这个！

注意：我们故意让这个答案错误，所以我们期望看到一个低分数。

In [ ]:
# 来自数据集示例
inputs = {
  "question": "LangSmith 是否与 LangChain 原生集成？"
}
reference_outputs = {
  "output": "是的，LangSmith 与 LangChain 以及 LangGraph 原生集成。"
}


# 来自运行
outputs = {
  "output": "不，LangSmith 没有与 LangChain 集成。"
}

similarity_score = compare_semantic_similarity(inputs, reference_outputs, outputs)
print(f"语义相似性分数: {similarity_score}")

您也可以直接使用 Run 和 Example 定义评估器！

In [ ]:
from langsmith.schemas import Run, Example

def compare_semantic_similarity_v2(root_run: Run, example: Example):
    input_question = example["inputs"]["question"]
    reference_response = example["outputs"]["output"]
    run_response = root_run["outputs"]["output"]
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {   
                "role": "system",
                "content": (
                    "您是一个语义相似性评估器。比较两个问题回答的含义，"
                    "参考回答和新回答，其中参考是正确答案，我们试图判断新回答是否相似。"
                    "提供1到10之间的分数，其中1表示完全不相关，10表示含义完全相同。"
                ),
            },
            {"role": "user", "content": f"问题: {input_question}\n 参考回答: {reference_response}\n 运行回答: {run_response}"}
        ],
        response_format=Similarity_Score,
    )

    similarity_score = completion.choices[0].message.parsed
    return {"score": similarity_score.similarity_score, "key": "similarity"}

In [ ]:
sample_run = {
  "name": "示例运行",
  "inputs": {
    "question": "LangSmith 是否与 LangChain 原生集成？"
  },
  "outputs": {
    "output": "不，LangSmith 没有与 LangChain 集成。"
  },
  "is_root": True,
  "status": "success",
  "extra": {
    "metadata": {
      "key": "value"
    }
  }
}

sample_example = {
  "inputs": {
    "question": "LangSmith 是否与 LangChain 原生集成？"
  },
  "outputs": {
    "output": "是的，LangSmith 与 LangChain 以及 LangGraph 原生集成。"
  },
  "metadata": {
    "dataset_split": [
      "AI生成",
      "基础"
    ]
  }
}

similarity_score = compare_semantic_similarity_v2(sample_run, sample_example)
print(f"语义相似性分数: {similarity_score}")